# 04 · Publication Bias Analysis

Detect and quantify publication bias in the LENR corpus using funnel plots and Egger's test.

> **[MOCK]** Effect size and standard error column names are placeholders.  
> Wire to real extracted parameters after physicist session defines the ontology.

In [ ]:
import os, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from dotenv import load_dotenv

load_dotenv()
API_BASE = os.getenv('VERITAS_API_BASE', 'http://localhost:5000')
CORPUS_ID = os.getenv('VERITAS_CORPUS_ID', 'demo-lenr-corpus')  # [MOCK]

In [ ]:
# [MOCK] EFFECT_COL and SE_COL — replace with real field names from LENR ontology
EFFECT_COL = 'cop'              # [MOCK] proxy for effect size (e.g. COP - 1 = excess)
SE_COL = 'extraction_confidence'  # [MOCK] proxy for precision; replace with real SE

resp = requests.get(f'{API_BASE}/api/corpora/{CORPUS_ID}/documents')
if resp.ok:
    documents = resp.json()
else:
    # [MOCK] Synthetic data with publication bias (positive results overrepresented)
    rng = np.random.default_rng(0)
    n = 40
    true_effect = 1.3
    se = rng.uniform(0.05, 0.4, n)
    # Introduce positive bias: suppress studies with negative results
    effect = true_effect + rng.normal(0, 1, n) * se
    effect = np.where(effect < 1.0, 1.0 + rng.uniform(0, 0.1, n), effect)  # [MOCK] bias
    documents = [
        {'document_id': f'doc-{i:03d}', 'extracted_fields': {
            'cop': float(effect[i]),
            'extraction_confidence': float(1.0 / (se[i] + 0.001))
        }} for i in range(n)
    ]
    print('[MOCK] Using synthetic biased data')

rows = [{'document_id': d['document_id'], **d.get('extracted_fields', {})} for d in documents]
df = pd.DataFrame(rows)
print(f'{len(df)} documents loaded')

In [ ]:
# Funnel plot: effect size vs precision
if EFFECT_COL in df.columns and SE_COL in df.columns:
    effect = df[EFFECT_COL].dropna()
    precision = df[SE_COL].dropna()
    
    mean_effect = effect.mean()
    
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(effect, precision, alpha=0.6, s=35)
    ax.axvline(mean_effect, color='red', linestyle='--', linewidth=1.5, label=f'Mean = {mean_effect:.2f}')
    ax.set_xlabel(f'Effect size ({EFFECT_COL})')
    ax.set_ylabel(f'Precision proxy ({SE_COL})')
    ax.set_title('Funnel Plot — Publication Bias Assessment')
    ax.legend()
    plt.tight_layout()
    plt.savefig('funnel_plot.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Egger's test (regression of effect size on precision)
# Significant intercept != 0 indicates publication bias
if EFFECT_COL in df.columns and SE_COL in df.columns:
    y = df[EFFECT_COL].dropna().values
    x = df[SE_COL].dropna().values[:len(y)]
    slope, intercept, r, p, se_slope = stats.linregress(x, y)
    print(f"Egger's test: intercept = {intercept:.3f}, p = {p:.4f}")
    if p < 0.05:
        print('⚠️  Significant asymmetry detected (p < 0.05) — possible publication bias')
    else:
        print('No significant funnel asymmetry detected')
    print(f'\n⚠️  All findings are correlational. Specific to corpus [{CORPUS_ID}].')
    print('    [MOCK] These results use synthetic data and are not scientifically valid.')